# Lab 13 — AlexNet: Filters, Feature Maps & Transfer Learning

*Open up a network that already works, look at what it learned, then retrain it on your own images.*

## Objectives

1. Load a pretrained AlexNet and inspect its layers and shapes
2. Visualize first-layer filters and intermediate feature maps
3. Preprocess arbitrary images into the tensor the network expects
4. Retrain the final layer on a dataset you assembled

## Interactive labs for this week

- [`alexnet.html`](../interactive/alexnet.html)

---

In [ ]:
# Run this first, every session.
!pip install -q git+https://github.com/williamedwardhahn/mpcr
from mpcr import *
import torch

# Every lab is device-agnostic: it runs on a GPU runtime or a plain CPU one.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Running on:', device)

#[Tensorspace](https://tensorspace.org/html/playground/alexnet.html)

![alt text](https://miro.medium.com/max/750/1*IlzW43-NtJrwqtt5Xy3ISA.jpeg)

![alt text](https://d3i71xaburhd42.cloudfront.net/38211dc39e41273c0007889202c69f841e02248a/2-Figure1-1.png)

![alt text](https://i.stack.imgur.com/XBWc3.png)

![alt text](https://cv-tricks.com/wp-content/uploads/2017/03/alexnet_small-1.png)

![alt text](https://www.learnopencv.com/wp-content/uploads/2017/11/convolution-example-matrix.gif)

<img src="https://production-media.paperswithcode.com/methods/MaxpoolSample2.png">

<img src=https://ujwlkarn.files.wordpress.com/2016/08/giphy.gif width="900">


In [ ]:
import matplotlib.pyplot as plt
import imageio
import torch
import torchvision
from torchvision import models, transforms
import numpy as np
from torchvision.models import *
from PIL import Image
import requests
from torchvision import models
from torchsummary import summary
from skimage.util import montage
from torch.autograd import Variable
from skimage import io as skio
from scipy import signal

In [ ]:
def scale(x):
    return (x - np.min(x))/np.max(x)
def scale1(x):
    return (x-np.min(x))/(np.max(x)-np.min(x))

In [ ]:
def plot(x):
    fig, ax = plt.subplots()
    im = ax.imshow(x,cmap='gray')
    ax.axis('off')
    fig.set_size_inches(10, 10)
    plt.show()

In [ ]:
image = skio.imread("http://ian-albert.com/games/super_mario_bros_maps/mario-2-2.gif")
image = image[:,0:700,:]
plot(image)

In [ ]:
coin = image[185:200,224:239,:]

In [ ]:
plot(coin)

In [ ]:
image = image[60:,0:700,:]
plot(image)

In [ ]:
image=np.mean(image,axis=2)
coin=np.mean(coin,axis=2)
image=scale1(image)
coin=scale1(coin)

In [ ]:
plot(image)


In [ ]:
plot(coin)

In [ ]:
coin.shape

In [ ]:
image = image - np.mean(image)
coin = coin - np.mean(coin)

In [ ]:
z=signal.convolve2d(image, np.rot90(coin, 2))

In [ ]:
plot(z)

In [ ]:
plot(z==np.max(z))

In [ ]:
[y,x] = np.where(z == np.amax(z))

In [ ]:
plt.plot(x,-y,'r.',markersize=30)
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(image, cmap = 'gray')
ax.axis('off')
fig.set_size_inches(20, 10)
ax.scatter(x-6, y-6, c='#f97306', s=80) #https://xkcd.com/color/rgb/
plt.show()

In [ ]:
net = alexnet(weights=AlexNet_Weights.DEFAULT).to(device)

In [ ]:
normalize = transforms.Normalize(
   mean=[0.485, 0.456, 0.406],
   std=[0.229, 0.224, 0.225]
)
preprocess = transforms.Compose([
   transforms.Resize(256),
   transforms.CenterCrop(224),
   transforms.ToTensor(),
   normalize
])

In [ ]:
im = imageio.imread('https://www.medicalnewstoday.com/content/images/articles/322/322868/golden-retriever-puppy.jpg')

In [ ]:
plot(im)

In [ ]:
image = Image.fromarray(im) #convert to pil

In [ ]:
img_tensor = preprocess(image)

In [ ]:
img_tensor = img_tensor.unsqueeze_(0)

In [ ]:
img_tensor.shape

In [ ]:
img_variable = img_tensor.to(device)

In [ ]:
img_variable.shape

In [ ]:
%%timeit
out = net(img_variable)

In [ ]:
out = net(img_variable)

In [ ]:
out.shape

In [ ]:
out

In [ ]:
label_index = out.cpu().data.numpy().argmax()

In [ ]:
label_index

In [ ]:
top_list = np.flip(np.argsort(out.cpu().data.numpy())[0][-10:])

In [ ]:
LABELS_URL = 'https://s3.amazonaws.com/mlpipes/pytorch-quick-start/labels.json'

In [ ]:
labels = {int(key):value for (key, value) in requests.get(LABELS_URL).json().items()}

In [ ]:
print(labels[label_index])

In [ ]:
for i in range(10):
    print(labels[top_list[i]])

In [ ]:
net

In [ ]:
summary(net, (3, 224, 224))

In [ ]:
out = net.features[0](img_variable).cpu().detach().numpy()

In [ ]:
plot(out[0,0,:,:])

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:6](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
im = imageio.imread('http://bocasurfcam.com/most_recent_image.php')

In [ ]:
plot(im)

In [ ]:
def load_im(im):
    image = Image.fromarray(im) #convert to pil
    img_tensor = preprocess(image)
    img_tensor = img_tensor.unsqueeze_(0)
    img_variable = img_tensor.to(device)
    return img_variable

In [ ]:
out = net(load_im(im))

In [ ]:
def inference(im):
    out = net(load_im(im))
    label_index = out.cpu().data.numpy().argmax()
    top_list = np.flip(np.argsort(out.cpu().data.numpy())[0][-10:])
    print(labels[label_index])
    print('____')
    for i in range(10):
        print(labels[top_list[i]])

In [ ]:
inference(im)

In [ ]:
net

In [ ]:
w1 = net.features[0].weight.data.cpu().numpy()

In [ ]:
w1.shape

In [ ]:
w1 = np.transpose(w1,(0,2,3,1))

In [ ]:
w1.shape

In [ ]:
Wp = (w1 - np.mean(w1))/np.std(w1)

In [ ]:
M = montage(Wp, multichannel=True)

In [ ]:
M.shape

In [ ]:
plot(Wp[0,:,:,:])

In [ ]:
plot(np.random.random((11,11,3)))

In [ ]:
plot(montage(Wp, multichannel=True))

In [ ]:
plot(montage(w1, multichannel=True))

In [ ]:
plot(scale1(montage(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1)), multichannel=True)))

In [ ]:
plot(scale(montage(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1)), multichannel=True)))

In [ ]:
img_variable.flatten().shape

In [ ]:
out = net.features[0](img_variable).cpu().detach().numpy()

In [ ]:
out.shape

In [ ]:
plot(out[0,0,:,:])

In [ ]:
img_variable.shape

In [ ]:
3*224*224

In [ ]:
64*55*55

In [ ]:
out = np.squeeze(out)

In [ ]:
plot(montage(out))

In [ ]:
plot(scale(montage(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1)), multichannel=True)))

In [ ]:
plot(montage(np.squeeze(net.features[0:1](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:2](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:3](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:4](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:5](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:6](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:7](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:8](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:9](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:10](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:11](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:12](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.avgpool(net.features[0:13](img_variable)).cpu().detach().numpy())))

In [ ]:
net.features[0:13](img_variable).cpu().detach().numpy().shape

In [ ]:
net

In [ ]:
net.avgpool(net.features[0:13](img_variable)).shape

In [ ]:
net.avgpool(net.features[0:13](img_variable)).flatten().shape[0]

In [ ]:
plt.plot(np.arange(9216),net.avgpool(net.features[0:13](img_variable)).flatten().cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
9216 / (3*224*224) # percentage of numbers left, vs original pixels

In [ ]:
net.classifier[0](net.avgpool(net.features[0:13](img_variable)).flatten()).shape

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:2](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:3](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:5](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:6](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(1000),net.classifier[0:7](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
out = net.classifier[0:7](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy()

In [ ]:
def softmax(x):

    s = np.exp(x)

    s = s / np.sum(s)

    return s

In [ ]:
out = softmax(out)

In [ ]:
plt.plot(np.arange(1000),out)
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
label_index = out.argmax()

In [ ]:
label_index

In [ ]:
top_list = np.flip(np.argsort(out))[0:10]

In [ ]:
top_list

In [ ]:
for i in range(10):
    print(labels[top_list[i]])

# Retrain

In [ ]:
Y = np.zeros(50,)
Y[25:] = 1

In [ ]:
Y

In [ ]:
X = y.detach().cpu().numpy()

In [ ]:
X.shape

In [ ]:
plt.plot(X[0],'.')

In [ ]:
X[0]

In [ ]:
np.argmax(X[0])

In [ ]:
index_winner = np.argmax(X[0])

labels[index_winner]

In [ ]:
top_ten = np.argsort(X[0])[::-1][0:10]

In [ ]:
for i in top_ten:
    print(labels[i])

In [ ]:
labels

In [ ]:
plt.hist(X[0])

In [ ]:
X = GPU_data(X)
Y = GPU_data(Y)

In [ ]:
X.shape

In [ ]:
!nvidia-smi

In [ ]:
def softmax(x):
    s1 = torch.exp(x - torch.max(x,1)[0][:,None])
    s = s1 / s1.sum(1)[:,None]
    return s

In [ ]:
def cross_entropy(outputs, labels):
    return -torch.sum(softmax(outputs).log()[range(outputs.size()[0]), labels.long()])/outputs.size()[0]

In [ ]:
def randn_trunc(s): #Truncated Normal Random Numbers
    mu = 0
    sigma = 0.1
    R = stats.truncnorm((-2*sigma - mu) / sigma, (2*sigma - mu) / sigma, loc=mu, scale=sigma)
    return R.rvs(s)

In [ ]:
def Truncated_Normal(size):

    u1 = torch.rand(size)*(1-np.exp(-2)) + np.exp(-2)
    u2 = torch.rand(size)
    z  = torch.sqrt(-2*torch.log(u1)) * torch.cos(2*np.pi*u2)

    return z

In [ ]:
def acc(out,y):
    with torch.no_grad():
        return (torch.sum(torch.max(out,1)[1] == y).item())/y.shape[0]

In [ ]:
X.shape

In [ ]:
Y.shape

In [ ]:
a = np.array([[1, 2], [3, 4]])

In [ ]:
b = np.array([[5, 6], [1, 0]])

In [ ]:
a.shape,b.shape

In [ ]:
a@b

In [ ]:
def inner_prod(a,b):
    c = 0
    for i in range(a.shape[0]):
        c += a[i]*b[i]
    return c

In [ ]:
def mat_mult(a,b):

    if a.shape[1] == b.shape[0]:

        c = np.zeros((a.shape[0],b.shape[1]))

        for i in range(a.shape[0]):
            for j in range(b.shape[1]):
               c[i,j] = inner_prod(a[i,:],b[:,j])
        return c

    else:
        print("error")



In [ ]:
%%timeit
a@b

In [ ]:
%%timeit
mat_mult(a,b)

In [ ]:
x = np.random.random((1000,1))

In [ ]:
w = np.random.random((2,1000))

In [ ]:
w.shape,x.shape

In [ ]:
z = mat_mult(w,x)

In [ ]:
z.shape

In [ ]:
z

In [ ]:
def get_batch(mode):
    b = c.b
    if mode == "train":
        r = np.random.randint(X.shape[0]-b)
        x = X[r:r+b,:]
        y = Y[r:r+b]
    elif mode == "test":
        r = np.random.randint(X_test.shape[0]-b)
        x = X_test[r:r+b,:]
        y = Y_test[r:r+b]
    return x,y

In [ ]:
def model(x,w):

    return x@w[0]

In [ ]:
def make_plots():

    acc_train = acc(model(x,w),y)

    # xt,yt = get_batch('test')

    # acc_test = acc(model(xt,w),yt)

    wb.log({"acc_train": acc_train})

In [ ]:
wb.init(project="Linear_Model_Photo_1");
c = wb.config

c.h = 0.001
c.b = 32
c.epochs = 100000

w = [GPU(Truncated_Normal((1000,2)))]

optimizer = torch.optim.Adam(w, lr=c.h)

for i in range(c.epochs):

    x,y = get_batch('train')

    loss = cross_entropy(softmax(model(x,w)),y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    wb.log({"loss": loss})

    make_plots()


https://tensorspace.org/

---

## Homework

1. Assemble a two-class image dataset of your own (30+ images per class) and retrain AlexNet on it.
2. Report accuracy and show the feature maps for one correct and one incorrect prediction.

Submit as a markdown report in your course GitHub repo, with plots committed alongside it.

<details><summary>Where this lab came from</summary>

- `_source/notebooks/MathData24/Feature_Maps__1GayNzpg.ipynb` cells 0–117 (116 kept)
- `_source/notebooks/MathDataSci_Fall23/Alexnet__1onlVrce.ipynb` cells 23–76 (42 kept)

</details>